In [1]:
%pip install -q librosa pandas numpy matplotlib soundfile kagglehub
%pip install -q pathlib
import kagglehub 

import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_76c34848cd62be4b168fbbc1eaa44ada'    

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import gc
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import timm
from tqdm import tqdm
from pathlib import Path

In [3]:
# 1. CONFIGURATION
class CFG:
    SR = 32000
    # ========== PSEUDO Labelling specific =========
    FORCE_REFRESH_SPEC_CACHE = True
    CHUNK_DURATION = 5
    HOP_DURATION = 5

    CHUNK_SIZE = SR * CHUNK_DURATION
    HOP_SIZE = int(SR * HOP_DURATION)
    
    TOP_K = 3

    PSEUDO_THRESHOLD = 0.20

    PRIMARY_LABEL_BOOST = 0.90    
    SECONDARY_LABEL_BOOST = 0.60    
    MIN_AUDIO_ENERGY = 0.001
    
    #=========
    DURATION = 5
    CHUNK_SIZE = SR * DURATION
    INPUT_DIR = Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes/")
   
    MODEL_PATH_B3_COMP_PRIMARY = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/composite-primary-labels/1/bird_model_best.pth"
    MODEL_PATH_B0 = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/syntheticdata/8/bird_model_best.pth"
   
    NUM_CLASSES = 234
    THRESHOLD = 0.00
    B3_COMP_PRI_WEIGHT = 0.85
    B0_WEIGHT = 0.15  # B0 is also secondary


In [4]:
# REFINED MODEL CLASS
class BirdModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', num_classes=234):
        super().__init__()
        # 1. Create model with 1 input channel instead of 3
        self.backbone = timm.create_model(model_name, pretrained=False, in_chans=1)
        
        # 2. Based on your error message, your trained weights use "fc" 
        # instead of "backbone.classifier". Let's map it:
        self.fc = nn.Linear(self.backbone.classifier.in_features, num_classes)
        
        # 3. Disable the default classifier so it doesn't conflict
        self.backbone.classifier = nn.Identity()
        
    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x) # Use the "fc" layer as in your training
        return x

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Load B3 Model trained on composite primary labels and no soundscape mixing (234 classes)
model_b3_comp_prim = BirdModel(model_name='efficientnet_b3', num_classes=234)
model_b3_comp_prim.load_state_dict(torch.load(CFG.MODEL_PATH_B3_COMP_PRIMARY, map_location=device))
model_b3_comp_prim.to(device).eval()


# Load B0 Model (206 classes)
model_b0 = BirdModel(model_name='efficientnet_b0', num_classes=206)
model_b0.load_state_dict(torch.load(CFG.MODEL_PATH_B0, map_location=device))
model_b0.to(device).eval()

Using device: cuda


BirdModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): Con

In [6]:
# 4. LOAD SUBMISSION FORMAT + CREATE SPECIES MAPPING

# Load the official sample submission to get the 234 required species columns
sample_sub_path = Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path)
target_species = sample_sub.columns[1:].tolist()

# the data is here:
data_root = Path("/kaggle/input/competitions/birdclef-2026")

# Load full 234 taxonomy
taxonomy_path = Path("/kaggle/input/competitions/birdclef-2026/taxonomy.csv")
taxonomy_df = pd.read_csv(taxonomy_path)
all_species_234 = sorted(taxonomy_df['primary_label'].unique())
mapping_234 = {s: i for i, s in enumerate(all_species_234)}

print(f"✅ Mapping created for {len(mapping_234)} species.")
# Verification check:
if len(mapping_234) != 234:
    print(f"⚠️ Warning: Found {len(mapping_234)} species, expected 234!")

# Load the 206 training labels used for B0
# Assuming you have the train.csv available
train_df = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train.csv")
species_206 = sorted(train_df['primary_label'].unique())

# Create an index map: which index in B3 (234) corresponds to the index in B0 (206)?
b0_to_b3_idx = [mapping_234[s] for s in species_206]

✅ Mapping created for 234 species.


In [7]:
def get_spectrogram(audio_chunk):
    # Matches the 1-channel logic your model expects
    S = librosa.feature.melspectrogram(y=audio_chunk, sr=CFG.SR, n_mels=128, fmax=16000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    return S_dB

In [8]:
# ============================================================
# 6. PSEUDO LABEL DATASET GENERATION (STREAMING ONLY)
# ============================================================

import pandas as pd
import numpy as np
import librosa
import torch
from pathlib import Path
from tqdm import tqdm
import gc

train_df = pd.read_csv(
    "/kaggle/input/competitions/birdclef-2026/train.csv"
)

TRAIN_AUDIO_DIR = Path(
    "/kaggle/input/competitions/birdclef-2026/train_audio"
)


idx_to_species_234 = {i: s for s, i in mapping_234.items()}

CFG.BATCH_SIZE = 64

def format_time(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def process_batch(spec_batch, meta_batch, row, primary_label, filename, manifest_rows):

    if len(spec_batch) == 0:
        return

    batch = torch.tensor(
        np.stack(spec_batch),
        dtype=torch.float32
    ).unsqueeze(1).to(device)

    with torch.inference_mode():
        out_b3 = model_b3_comp_prim(batch)
        probs_b3 = torch.sigmoid(out_b3)

        out_b0 = model_b0(batch)
        probs_b0 = torch.sigmoid(out_b0)

        probs_b0_full = torch.zeros((len(spec_batch), 234), device=device)
        probs_b0_full[:, b0_to_b3_idx] = probs_b0

        blended = (
            probs_b3 * CFG.B3_COMP_PRI_WEIGHT +
            probs_b0_full * CFG.B0_WEIGHT
        ).cpu().numpy()

    secondary = row.get("secondary_labels", "[]")

    try:
        if isinstance(secondary, str):
            secondary = eval(secondary)
    except:
        secondary = []

    for b, meta in zip(blended, meta_batch):

        # primary boost
        if primary_label in mapping_234:
            i = mapping_234[primary_label]
            b[i] = max(b[i], CFG.PRIMARY_LABEL_BOOST)

        # secondary boost
        for s in secondary:
            if s in mapping_234:
                j = mapping_234[s]
                b[j] = max(b[j], CFG.SECONDARY_LABEL_BOOST)

        b = np.power(b, 0.90)

        topk = np.argsort(b)[::-1][:CFG.TOP_K]

        labels = []
        for i in topk:
            if b[i] >= CFG.PSEUDO_THRESHOLD:
                labels.append(idx_to_species_234[i])

        if primary_label not in labels:
            labels.append(primary_label)

        labels = list(set(labels))
        if len(labels) == 0:
            continue

        manifest_rows.append({
            "filename": filename,
            "start": format_time(meta["start"] / CFG.SR),
            "end": format_time(meta["end"] / CFG.SR),
            "primary_label": ";".join(labels)
        })


def run_pseudo_label_generation(train_df, TRAIN_AUDIO_DIR):
    
    manifest_rows = []

    for _, row in tqdm(train_df.iterrows(), total=len(train_df)):

        primary_label = row["primary_label"]
        filename = row["filename"]

        audio_path = TRAIN_AUDIO_DIR / filename
        if not audio_path.exists():
            continue

        try:
            y, _ = librosa.load(audio_path, sr=CFG.SR)

            if len(y) <= CFG.CHUNK_SIZE:
                starts = [0]
            else:
                starts = range(0, len(y) - CFG.CHUNK_SIZE + 1, CFG.HOP_SIZE)

            spec_batch, meta_batch = [], []
            chunk_index = 0

            for start in starts:

                end = start + CFG.CHUNK_SIZE
                chunk = y[start:end]

                if len(chunk) < CFG.CHUNK_SIZE:
                    chunk = np.pad(chunk, (0, CFG.CHUNK_SIZE - len(chunk)))

                if np.abs(chunk).mean() < CFG.MIN_AUDIO_ENERGY:
                    continue

                spec = get_spectrogram(chunk)

                spec_batch.append(spec)
                meta_batch.append({
                    "chunk_index": chunk_index,
                    "start": start,
                    "end": min(end, len(y))
                })

                chunk_index += 1

                if len(spec_batch) >= CFG.BATCH_SIZE:
                    process_batch(
                        spec_batch, meta_batch,
                        row, primary_label, filename,
                        manifest_rows
                    )
                    spec_batch, meta_batch = [], []

            process_batch(
                spec_batch, meta_batch,
                row, primary_label, filename,
                manifest_rows
            )

            del y
            gc.collect()

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    return manifest_rows  

In [9]:
from pathlib import Path
import shutil
import json
import os
import gc
import pandas as pd
from datetime import datetime


def generate_pseudo_labels_if_needed():

    output_path = Path("/kaggle/working/pseudo_labels.csv")

    dataset_file_path = Path(
        "/kaggle/input/datasets/gany24558/chunked-pseudo-labels-birdclef2026/pseudo_labels.csv"
    )

    dataset_dir = Path("/kaggle/working/pseudo_soundscape_dataset")
    dataset_dir.mkdir(parents=True, exist_ok=True)

    dataset_id = "gany24558/chunked-pseudo-labels-birdclef2026"

    # -----------------------------
    # STEP 1: Check Kaggle dataset (source of truth)
    # -----------------------------
    if dataset_file_path.exists():
        print("✅ Found pseudo_labels.csv in Kaggle dataset → skipping generation")
        return None

    print("🚀 File not found in Kaggle dataset → generating pseudo labels...")

    train_df = pd.read_csv(
        "/kaggle/input/competitions/birdclef-2026/train.csv"
    )

    #train_df = train_df.head(100)

    train_audio_dir = Path(
        "/kaggle/input/competitions/birdclef-2026/train_audio"
    )

    # -----------------------------
    # STEP 2: Generate pseudo labels
    # -----------------------------
    manifest_rows = run_pseudo_label_generation(
        train_df,
        train_audio_dir
    )

    pseudo_df = pd.DataFrame(manifest_rows)

    # Atomic write (important safety improvement)
    tmp_path = output_path.with_suffix(".tmp.csv")
    pseudo_df.to_csv(tmp_path, index=False)
    tmp_path.replace(output_path)

    print(f"✅ Saved locally: {output_path}")
    print(f"✅ Total chunks: {len(pseudo_df):,}")

    # -----------------------------
    # STEP 3: Prepare Kaggle dataset
    # -----------------------------
    shutil.copy2(output_path, dataset_dir / "pseudo_labels.csv")

    metadata = {
        "title": f"chunked-pseudo-labels-birdclef2026-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
        "id": dataset_id,
        "licenses": [{"name": "CC0-1.0"}]
    }

    (dataset_dir / "dataset-metadata.json").write_text(
        json.dumps(metadata, indent=2)
    )

    # -----------------------------
    # STEP 4: Upload / version dataset
    # -----------------------------
    print("🚀 Uploading to Kaggle...")

    exit_code = os.system(
        f"kaggle datasets version -p {dataset_dir} -m \"Updated pseudo labels\""
    )

    if exit_code != 0:
        print("⚠️ Kaggle upload failed (check CLI/auth/metadata)")

    gc.collect()

    print("✅ Done")

    return pseudo_df


pseudo_df = generate_pseudo_labels_if_needed()

✅ Found pseudo_labels.csv in Kaggle dataset → skipping generation


# End of Pseudo-Labelling

In [10]:
import os
def upload_spectrogram_cache(
    cache_dir,
    dataset_name,
    username,
) -> str:
    """
    Writes Kaggle dataset metadata and uploads the spectrogram
    cache directory as a new (or updated) Kaggle dataset via kagglehub.

    Args:
        cache_dir:    Directory containing the cache to upload.
        dataset_name: Slug for the Kaggle dataset.
        username:     Kaggle username. Falls back to the
                      KAGGLE_USERNAME environment variable.

    Returns:
        The dataset handle string  "<username>/<dataset_name>".

    Raises:
        ValueError: If no username is resolved.
    """

    user = username or os.environ.get("KAGGLE_USERNAME")
    if not user:
        raise ValueError(
            "No Kaggle username provided. Pass `username=` or set "
            "the KAGGLE_USERNAME environment variable."
        )

    handle = f"{user}/{dataset_name}"

    # --------------------------------------------------------
    # WRITE METADATA
    # --------------------------------------------------------

    metadata = {
        "title":    dataset_name,
        "id":       handle,
        "licenses": [{"name": "CC0-1.0"}],
    }

    meta_path = cache_dir / "dataset-metadata.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Metadata written: {meta_path}")

    # --------------------------------------------------------
    # UPLOAD VIA KAGGLEHUB
    # --------------------------------------------------------

    url = kagglehub.dataset_upload(
        handle=handle,
        local_dataset_dir=str(cache_dir),
    )

    print(f"Upload complete: {url}")
    return handle




In [11]:
import os
import librosa
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

# ============================================================
# PATHS
# ============================================================

PSEUDO_LABELS_CSV = (
    "/kaggle/input/datasets/"
    "gany24558/chunked-pseudo-labels-birdclef2026/"
    "pseudo_labels.csv"
)

TRAIN_AUDIO_DIR = Path(
    "/kaggle/input/competitions/birdclef-2026/train_audio"
)

CACHE_DIR = Path("/kaggle/working/spectrogram_cache")
SPEC_DIR = CACHE_DIR / "specs"


# ============================================================
# HELPERS
# ============================================================

def time_to_seconds(x):
    h, m, s = x.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


# ============================================================
# MAIN METHOD
# ============================================================

def build_spectrogram_cache(
    pseudo_labels_csv: str = PSEUDO_LABELS_CSV,
    train_audio_dir: Path = TRAIN_AUDIO_DIR,
    cache_dir: Path = CACHE_DIR,
    debug_rows: int | None = None,
) -> pd.DataFrame:
    """
    Loads pseudo-labelled audio chunks, computes mel spectrograms,
    saves them as .npy files, and writes a manifest CSV.

    Args:
        pseudo_labels_csv:  Path to the pseudo labels CSV.
        train_audio_dir:    Root directory of training audio files.
        cache_dir:          Directory where specs/ and the manifest are saved.
        debug_rows:         If set, only process the first N rows (debug mode).

    Returns:
        DataFrame with columns [spec_path, primary_label].
    """

    spec_dir = cache_dir / "specs"
    cache_dir.mkdir(parents=True, exist_ok=True)
    spec_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # LOAD & OPTIONALLY SUBSET
    # --------------------------------------------------------

    df = pd.read_csv(pseudo_labels_csv)
    print(f"Total pseudo rows: {len(df):,}")

    if debug_rows is not None:
        df = df.head(debug_rows)
        print(f"Debug mode: using first {debug_rows} rows")

    grouped = df.groupby("filename")
    print(f"Unique audio files: {len(grouped):,}")

    # --------------------------------------------------------
    # CACHE GENERATION
    # --------------------------------------------------------

    cached_rows = []

    for filename, group in tqdm(grouped, total=len(grouped)):

        audio_path = train_audio_dir / filename
        if not audio_path.exists():
            continue

        try:
            y, _ = librosa.load(audio_path, sr=CFG.SR)

            for idx, row in group.iterrows():

                start_sec = time_to_seconds(row["start"])
                end_sec   = time_to_seconds(row["end"])

                start = int(start_sec * CFG.SR)
                end   = int(end_sec   * CFG.SR)

                chunk = y[start:end]

                # Pad short chunks
                if len(chunk) < CFG.CHUNK_SIZE:
                    chunk = np.pad(chunk, (0, CFG.CHUNK_SIZE - len(chunk)))

                # Build a filesystem-safe name
                safe_name = filename.replace("/", "_").replace(".ogg", "")
                spec_name = f"{safe_name}_{idx}.npy"
                save_path = spec_dir / spec_name

                # Resume support — skip if already cached
                if not save_path.exists():
                    spec = get_spectrogram(chunk).astype(np.float16)
                    np.save(save_path, spec)

                cached_rows.append({
                    "spec_path":     spec_name,
                    "primary_label": row["primary_label"],
                })

            del y

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # --------------------------------------------------------
    # SAVE MANIFEST
    # --------------------------------------------------------

    cached_df = pd.DataFrame(cached_rows)
    manifest_path = cache_dir / "cached_labels.csv"
    cached_df.to_csv(manifest_path, index=False)

    # --------------------------------------------------------
    # FINAL STATS
    # --------------------------------------------------------

    cache_size_gb = (
        sum(f.stat().st_size for f in spec_dir.glob("*.npy")) / (1024 ** 3)
    )

    print("=" * 48)
    print("Done caching spectrograms")
    print(f"Total cached: {len(cached_df):,}")
    print(f"Manifest:     {manifest_path}")
    print(f"Specs dir:    {spec_dir}")
    print(f"Cache size:   {cache_size_gb:.2f} GB")
    print("=" * 48)

    # --------------------------------------------------------
    # UPLOAD TO KAGGLE
    # --------------------------------------------------------

    upload_spectrogram_cache(cache_dir=cache_dir, 
                             dataset_name='chunked-spectrogram-cache-5s',
                             username='gany24558'
                            )


    return cached_df

In [12]:

# ============================================================
# ENTRY POINT for getting spectrogram cache
# ============================================================
if CFG.FORCE_REFRESH_SPEC_CACHE:
    cached_df = build_spectrogram_cache()
else:    
    try:
        cache_path = Path(kagglehub.dataset_download("gany24558/chunked-spectrogram-cache-5s"))
        cached_df = pd.read_csv(cache_path / "cached_labels.csv")
        print(f"Cache loaded from Kaggle: {cache_path}")
    except Exception:
        cached_df = build_spectrogram_cache(debug_rows=100)

Total pseudo rows: 226,037
Unique audio files: 34,889


100%|██████████| 34889/34889 [1:21:16<00:00,  7.16it/s]  


Done caching spectrograms
Total cached: 226,037
Manifest:     /kaggle/working/spectrogram_cache/cached_labels.csv
Specs dir:    /kaggle/working/spectrogram_cache/specs
Cache size:   16.89 GB
Metadata written: /kaggle/working/spectrogram_cache/dataset-metadata.json
Uploading Dataset https://api.kaggle.com/datasets/gany24558/chunked-spectrogram-cache-5s ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpnen2izuc/archive.zip


Uploading: 100%|██████████| 18.2G/18.2G [03:22<00:00, 89.9MB/s]

Upload successful: /tmp/tmpnen2izuc/archive.zip (17GB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/gany24558/chunked-spectrogram-cache-5s
Upload complete: None


In [13]:
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset


class BirdDataset(Dataset):

    def __init__(self, df, spec_dir, label_map):

        self.df = df.reset_index(drop=True)
        self.spec_dir = Path(spec_dir)

        self.label_map = label_map
        self.num_classes = len(label_map)

    def __len__(self):
        return len(self.df)

    def encode_labels(self, label_str):

        y = np.zeros(self.num_classes, dtype=np.float32)

        for lbl in str(label_str).split(";"):
            if lbl in self.label_map:
                y[self.label_map[lbl]] = 1.0

        return y

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Load precomputed spectrogram
        spec_path = self.spec_dir / row["spec_path"]

        spec = np.load(spec_path)

        # (128, T) -> (1, 128, T)
        spec = spec[np.newaxis, ...]

        spec = torch.tensor(
            spec,
            dtype=torch.float32
        )

        # Multi-label target
        label = self.encode_labels(row["primary_label"])

        label = torch.tensor(
            label,
            dtype=torch.float32
        )

        return spec, label

In [14]:
pseudo_df = pd.read_csv(
    "/kaggle/working/spectrogram_cache/cached_labels.csv"
)

species = sorted(mapping_234.keys())

label_map = {
    s: i
    for i, s in enumerate(species)
}

In [15]:
print(pseudo_df.columns.tolist())
pseudo_df.head()

['spec_path', 'primary_label']


,spec_path,primary_label
0,1161364_iNat1114648_3.npy,1161364
1,1161364_iNat1114648_4.npy,1161364
2,1161364_iNat1114648_5.npy,1161364
3,1161364_iNat1114648_6.npy,1161364
4,1161364_iNat1114648_7.npy,1161364


In [16]:
from sklearn.model_selection import GroupShuffleSplit
from pathlib import Path

# ============================================================
# TRAIN / VAL SPLIT
# GROUPED BY ORIGINAL AUDIO FILE (from spec_path)
# ============================================================

# Extract original audio id (removes chunk suffix like _3, _4, etc.)
pseudo_df["audio_id"] = pseudo_df["spec_path"].apply(
    lambda x: "_".join(Path(x).stem.split("_")[:-1])
)

splitter = GroupShuffleSplit(
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        pseudo_df,
        groups=pseudo_df["audio_id"]
    )
)

train_df = pseudo_df.iloc[train_idx].reset_index(drop=True)
val_df = pseudo_df.iloc[val_idx].reset_index(drop=True)

print(f"Train samples: {len(train_df):,}")
print(f"Val samples:   {len(val_df):,}")

print(f"Train audio files: {train_df['audio_id'].nunique():,}")
print(f"Val audio files:   {val_df['audio_id'].nunique():,}")

Train samples: 183,173
Val samples:   42,864
Train audio files: 27,911
Val audio files:   6,978


In [17]:
train_dataset = BirdDataset(
    train_df,
    "/kaggle/working/spectrogram_cache/specs",
    label_map
)

val_dataset = BirdDataset(
    val_df,
    "/kaggle/working/spectrogram_cache/specs",
    label_map
)

In [18]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

In [19]:
class BirdModel(nn.Module):
    def __init__(self, num_classes):
        super(BirdModel, self).__init__()
        # 1. Load a pre-trained EfficientNet-B0
        # in_chans=1 because our spectrograms are grayscale (1 channel)
        self.backbone = timm.create_model('efficientnet_b3', 
                                          pretrained=True, 
                                          num_classes=0, 
                                          in_chans=1)
        
        # 2. Get the number of features from the backbone output
        num_features = self.backbone.num_features
        
        # 3. Define the final classification head
        self.fc = nn.Linear(num_features, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        return x

# Initialize the model
#device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# Universal device selector: prioritizes NVIDIA GPU, then Apple Silicon, then CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else 
    "mps" if torch.backends.mps.is_available() else 
    "cpu"
)

# Optional but recommended for NVIDIA GPUs: optimizes convolution algorithms
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"Using device: {device}")

model = BirdModel(num_classes=234).to(device)

print(f"Model initialized on: {device}")

Using device: cuda


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Model initialized on: cuda


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
import os

# =========================
# 1. Hyperparameters
# =========================
EPOCHS = 50

# Max training limit — early stopping will likely trigger before this

EARLY_STOPPING_PATIENCE = 6
# Increased from 5 → gives the model more room to improve before stopping
# Important because LR is still high in early epochs, so temporary plateaus are expected

EARLY_STOPPING_MIN_DELTA = 1e-4
# Minimum improvement to reset patience counter
# Filters out noise — tiny fluctuations don't count as real improvement

# =========================
# 2. Loss and Optimizer
# =========================
criterion = nn.BCEWithLogitsLoss()
# Correct loss for multi-label classification + mixup
# Expects raw logits (not sigmoid-ed outputs)


optimizer = optim.Adam([
    {"params": model.backbone.parameters(), "lr": 1e-5},
    # Small LR for pretrained backbone — preserves learned ImageNet features
    # Slightly increased from 1e-5 to allow faster adaptation to bird spectrograms

    {"params": model.fc.parameters(), "lr": 1e-4},
    # Larger LR for classifier head — trained from scratch so can afford bigger steps
    # Maintains 10x ratio over backbone LR (standard best practice)
])

# =========================
# 3. Scheduler
# =========================
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    # First cosine cycle runs for 10 epochs — LR decays from max → eta_min
    # Aligns with expected early stopping window (~6-10 epochs)

    T_mult=1,
    # Each restart uses the same cycle length (10 epochs)
    # If training continues past epoch 10, the LR resets and decays again
    # Prevents the flat eta_min problem of plain CosineAnnealingLR

    eta_min=1e-6
    # Floor LR — allows fine-grained weight updates at end of each cycle
)

In [21]:
# =========================
# 3. Training Loop
# =========================
print(f"Starting training for {EPOCHS} epochs...")
best_val_loss = float('inf')
save_dir = "soundscape_v1_may_4"
os.makedirs(save_dir, exist_ok=True)

# Early stopping state
epochs_no_improve = 0
early_stopped = False

for epoch in range(EPOCHS):
    # -------------------------
    # TRAINING PHASE
    # -------------------------
    model.train()
    running_loss = 0.0
    current_lr = scheduler.get_last_lr()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train] LR={current_lr}")

    for i, (inputs, labels) in enumerate(pbar):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=running_loss / (i + 1))

    avg_train_loss = running_loss / len(train_loader)
    scheduler.step()
    
    # -------------------------
    # VALIDATION PHASE
    # -------------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for val_inputs, val_labels in val_loader:
            val_inputs = val_inputs.to(device)
            val_labels = val_labels.to(device)
            val_outputs = model(val_inputs)
            v_loss = criterion(val_outputs, val_labels).mean()
            val_loss += v_loss.item()

    avg_val_loss = val_loss / len(val_loader)

    # -------------------------
    # EARLY STOPPING CHECK
    # -------------------------
    if avg_val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
        # Genuine improvement — reset patience counter
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        best_save_path = os.path.join(save_dir, "bird_model_best.pth")
        torch.save(model.state_dict(), best_save_path)
        print(f"  ✅ Val loss improved → {best_val_loss:.4f} | Saved Best Model")
        kagglehub.model_upload(
            'gany24558/birdclef-apr-20-2026-gc/PyTorch/backup', 
            'soundscape_v1_may_4/', 
            version_notes=f'Epoch {epoch+1} — val loss {best_val_loss:.4f}'
        )
    else:
        # No improvement
        epochs_no_improve += 1
        print(f"  ⚠️ No improvement for {epochs_no_improve}/{EARLY_STOPPING_PATIENCE} epochs")
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"\n🛑 Early stopping triggered at epoch {epoch+1}!")
            early_stopped = True

    # -------------------------
    # EPOCH SUMMARY
    # -------------------------
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f} | Val Loss = {avg_val_loss:.4f} | LR = {scheduler.get_last_lr()}")

    if early_stopped:
        break

# =========================
# 4. Save Final Model
# =========================
save_path = os.path.join(save_dir, "bird_model_final.pth")
torch.save(model.state_dict(), save_path)
print(f"\nFinal model saved at: {save_path}")

if early_stopped:
    print(f"Training stopped early at epoch {epoch+1}/{EPOCHS}")
    print(f"Best model (val loss: {best_val_loss:.4f}) saved at: {best_save_path}")
else:
    print(f"Training completed all {EPOCHS} epochs")
    print(f"Best val loss: {best_val_loss:.4f} | Best model saved at: {best_save_path}")
    
# =========================
# 5. UPLOAD BEST MODEL TO KAGGLE
# =========================
print(f"\nUploading best model to Kaggle...")
kagglehub.model_upload(
    'gany24558/birdclef-apr-20-2026-gc/PyTorch/chunked-pseudo-labels',
    save_dir,
    version_notes=f'B3 + 234 classes chunked-pseudo-labels — best val loss {best_val_loss:.4f}'
)
print(f"✅ Upload complete!")
print(f"Model available at: https://www.kaggle.com/models/gany24558/birdclef-apr-20-2026-gc/PyTorch/chunked-pseudo-labels")        

Starting training for 50 epochs...


Epoch 1/50 [Train] LR=[1e-05, 0.0001]:   0%|          | 0/2863 [00:00<?, ?it/s]

  ✅ Val loss improved → 0.0326 | Saved Best Model
Uploading Model https://api.kaggle.com/models/gany24558/birdclef-apr-20-2026-gc/PyTorch/backup ...
Starting upload for file soundscape_v1_may_4/bird_model_best.pth


Uploading: 100%|██████████| 44.8M/44.8M [00:00<00:00, 103MB/s]

Upload successful: soundscape_v1_may_4/bird_model_best.pth (43MB)


...

Uploading: 100%|██████████| 44.8M/44.8M [00:00<00:00, 99.0MB/s]

Upload successful: soundscape_v1_may_4/bird_model_final.pth (43MB)
Starting upload for file soundscape_v1_may_4/bird_model_best.pth



Uploading: 100%|██████████| 44.8M/44.8M [00:00<00:00, 66.5MB/s]

Upload successful: soundscape_v1_may_4/bird_model_best.pth (43MB)


Your model instance version has been created.
Files are being processed...
See at: https://api.kaggle.com/models/gany24558/birdclef-apr-20-2026-gc/PyTorch/chunked-pseudo-labels
✅ Upload complete!
Model available at: https://www.kaggle.com/models/gany24558/birdclef-apr-20-2026-gc/PyTorch/chunked-pseudo-labels
